In [8]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from Bio import SeqIO,AlignIO
from Bio.Align import PairwiseAligner
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from matplotlib.colors import LogNorm
%pip install seaborn
import seaborn as sns

In [11]:
def get_identity(records):
    aligner = PairwiseAligner()
    aligner.mode = 'global'
    identity = []
    star_record = records[0]
    for record in records:
        alignments = aligner.align(star_record.seq, record.seq)
        best_alignment = alignments[0]
        alignment_length = max(len(record.seq),len(star_record.seq))
        identity.append(best_alignment.counts().identities/alignment_length)
        #print('identity: ',identity,alignment_length)
        #aligned_sequence = best_alignment.target
    return identity

def distance_matrix(seqs):
    aligner = PairwiseAligner()
    aligner.mode = 'global'
    n = len(seqs)
    dist = np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            aln = aligner.align(seqs[i].seq,seqs[j].seq)[0].counts().identities
            aln_length = max(len(seqs[i].seq),len(seqs[j].seq))
            dist[i,j] = 1-aln/aln_length
            dist[j,i] = 1-aln/aln_length
    #print(dist)
    return dist

def heatmap_distance_matrix(seqs,distances):
    plt.figure(figsize=(20,16))
    labels = [label.id for label in seqs]
    sns.heatmap(distances,annot=False,cmap="viridis",xticklabels=labels,yticklabels=labels)
    plt.title("Distance Matrix")
    plt.savefig("prova_matrix.pdf")
    plt.close()
    return None

def dendro(dist,input_seqs):
    Z = linkage(squareform(dist), method='average')
    plt.figure(figsize=(12,4))
    dendrogram(
        Z,
        labels=[i.id for i in input_seqs]
    )
    plt.ylabel("Distance")
    plt.savefig("prova_dendro2.pdf")

    return None

def conserved_plots(MSA,interesting_residues):
    conservation = []
    gap_fraction = []
    for pos in range(alignment.get_alignment_length()):
        column = alignment[:, pos]
        gaps = column.count('-')
        gap_fraction.append(gaps / len(column))
        residues = [r for r in column if r != '-']
        if len(residues) == 0:
            conservation.append(np.nan)
            continue
        score = (
            Counter(residues)
            .most_common(1)[0][1]
            / len(residues)
        )
        conservation.append(score)

    plt.figure(figsize=(12,4))
    plt.plot(conservation, label='Conservation')
    #plt.plot(gap_fraction, label='Gap fraction')
    for pos in interesting_residues:#[14,17,18,80]:#14,17]:
        plt.axvline(
            pos,
            color="red",
            alpha=0.5
        )
    plt.legend()
    plt.xlabel("Alignment Position")
    plt.ylabel("Conservation")
    plt.savefig("Conservation_plot.pdf")
    plt.close()
    
    #window = 1
    #smoothed = np.convolve(conservation,np.ones(window)/window,mode='same')
    #plt.figure(figsize=(12,4))
    #plt.plot(smoothed, label='Smoothed-conservations')
    #for pos in [14,17]:#[65,66,67,96,222]:
    #    plt.axvline(
    #        pos,
    #        color="red",
    #        alpha=0.5
    #    )
    #plt.savefig("Smoothed_cons.pdf")
    #plt.close()
    return conservation

def mutual_information(col1, col2):
    residues1 = set(col1)
    residues2 = set(col2)
    mi = 0
    for r1 in residues1:
        for r2 in residues2:
            pxy = sum(
                (a == r1 and b == r2)
                for a,b in zip(col1,col2)
            ) / len(col1)
            if pxy == 0:
                continue
            px = col1.count(r1)/len(col1)
            py = col2.count(r2)/len(col2)
            mi += pxy*np.log2(
                pxy/(px*py)
            )
    return mi

def coevolution_matrix():
    L = alignment.get_alignment_length()
    MI = np.zeros((L,L))
    for i in range(L):
        col_i = list(alignment[:,i])
        for j in range(i+1,L):
            col_j = list(alignment[:,j])
            MI[i,j] = mutual_information(
                col_i,
                col_j
            )
            MI[j,i] = MI[i,j]
#    L = alignment.get_alignment_length()
#    cov = np.zeros((L,L))
#    for i in range(L):
#        col_i = alignment[:,i]
#        for j in range(i+1,L):
#            col_j = alignment[:,j]
#            score = sum(a == b for a,b in zip(col_i,col_j)) / len(col_i)
#            cov[i,j] = score
#            cov[j,i] = score
    plt.figure(figsize=(14,12))
    sns.heatmap(
        MI,
        cmap="viridis"
    )

    plt.title("Mini AlphaFold Coevolution Matrix")
    plt.savefig("mini_coevolution_matrix.pdf")

    ## NORM ##
    threshold = np.percentile(MI, 97.0)
    plt.figure(figsize=(14,12))
    MI_sparse = np.where(MI>threshold,MI,np.nan)
    sns.heatmap(
        MI_sparse,
        cmap="viridis",
        norm=LogNorm()
    )

    plt.title("Mini AlphaFold Coevolution Matrix")
    plt.savefig("mini_coevolution_matrix_norm.pdf")

    #pairs = []
    #for i in range(MI.shape[0]):
    #    for j in range(i+10, MI.shape[1]):  # skip nearby residues
    #        if MI[i,j] > 0:
    #            pairs.append((MI[i,j], i, j))
    #pairs.sort(reverse=True)
    #for score, i, j in pairs[:20]:
    #    print(f"{i} <-> {j}: {score:.3f}")
    plt.close()
    return MI

